# 04 — Environment
Modificação da cena física em tempo de execução via `PUT /environment`.

## Pré-requisito
```bash
cd managing && python main.py
```

## Ações disponíveis
| `action`          | Descrição                            | Campos obrigatórios                 |
|-------------------|--------------------------------------|-------------------------------------|
| `move_obstacle`   | Move um obstáculo existente          | `name`, `position`                  |
| `add_obstacle`    | Adiciona um novo obstáculo           | `name`, `size`, `position`, `color` |
| `remove_obstacle` | Remove um obstáculo da cena          | `name`                              |
| `move_object`     | Teleporta um objeto manipulável      | `name`, `position`                  |
| `move_robot_base` | Reposiciona a base do braço robótico | `position`                          |

> **`position`**: `[x, y, z]` em metros  
> **`size`**: `[x, y, z]` tamanho total (não half-extents)  
> **`color`**: `[r, g, b, a]` entre 0.0 e 1.0

In [1]:
%pip install requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import requests
import json
import time

BASE_URL = "http://localhost:8000"

def get(endpoint: str) -> dict:
    r = requests.get(f"{BASE_URL}{endpoint}", timeout=3)
    r.raise_for_status()
    return r.json()

def put(endpoint: str, body: dict) -> dict:
    r = requests.put(f"{BASE_URL}{endpoint}", json=body, timeout=3)
    r.raise_for_status()
    return r.json()

def pp(data: dict):
    print(json.dumps(data, indent=2, default=str))

print("Setup OK")

Setup OK


---
## Leitura do estado atual da cena

In [ ]:
# Lista obstáculos com posição atual
pp(get("/environment/obstacles"))

In [ ]:
# Lista objetos manipuláveis com posição atual
pp(get("/environment/objects"))

---
## `move_obstacle` — mover obstáculo existente

In [ ]:
response = put("/environment", {
    "action":   "move_obstacle",
    "name":     "parede_1",
    "position": [0.09, 0.0, 0.04]
})
print(response)

---
## `add_obstacle` — adicionar novo obstáculo
O `name` deve ser único — não pode conflitar com nomes já existentes.

In [ ]:
response = put("/environment", {
    "action":   "add_obstacle",
    "name":     "parede_2",
    "size":     [0.02, 0.30, 0.08],
    "position": [0.09, 0.15, 0.04],
    "color":    [0.8, 0.5, 0.1, 0.9],
    "mass":     0.0
})
print(response)

---
## `remove_obstacle` — remover obstáculo
Após remover, o nome fica disponível para `add_obstacle`.

In [ ]:
response = put("/environment", {
    "action": "remove_obstacle",
    "name":   "parede_2"
})
print(response)

---
## `move_object` — teleportar o cubo
Reposiciona instantaneamente sem reiniciar o episódio.

In [ ]:
response = put("/environment", {
    "action":   "move_object",
    "name":     "cube_1",
    "position": [0.03, 0.0, 0.02]
})
print(response)

---
## `move_robot_base` — reposicionar a base do braço
O controle EE usa IK em coordenadas do mundo — o braço se adapta automaticamente à nova base no próximo step.

> **Posição padrão**: `[-0.4, 0.0, 0.0]`  
> **Atenção**: se mover durante as fases `LIFT`/`RELEASE` do pick-and-place, o braço tentará alcançar
> o target fixado antes do movimento — pode não conseguir se a nova base estiver muito longe.

In [14]:
# Mover base 20 cm para frente
response = put("/environment", {
    "action":   "move_robot_base",
    "position": [-0.5, 0.0, 0.0]
})
print(response)

{'ok': True}


In [19]:


# Restaurar posição original
response = put("/environment", {
    "action":   "move_robot_base",
    "position": [-0.5, 0.0, 0.0]
})
print(response)

{'ok': True}
